# cAST-Scope — run réel sur GPU (StarCoder2-7B / CodeLlama-7B)

Compare les 3 baselines de chunking (`fixed`, `cast_orig`, `cast_scope`) sur RepoEval et CrossCodeEval, avec le même retriever BM25 et le même générateur pour les 3 — voir `Robertkiza0/cAST-state` (https://github.com/Robertkiza0/cAST-state).

**v2** : toutes les cellules d'installation sont idempotentes (re-exécuter une cellule ne recrée pas de clone imbriqué ni ne retélécharge si déjà présent), avec un test minimal du générateur isolé du harnais complet AVANT le run à 300 tâches, pour repérer un problème en secondes plutôt qu'en devant attendre toute une boucle.

## 0. Vérifier le GPU

In [ ]:
!nvidia-smi

## 1. Cloner (ou mettre à jour) le dépôt cAST-state + installer les dépendances

Idempotent : si `/content/cAST-state` existe déjà (re-exécution de cette cellule, ou session reprise), on fait `git pull` au lieu de re-cloner par-dessus (c'est ça qui créait les dossiers imbriqués `cAST-state/cAST-state/...` dans la v1).

In [ ]:
import os

if os.path.isdir('/content/cAST-state'):
    %cd /content/cAST-state
    !git pull
else:
    !git clone --depth 1 https://github.com/Robertkiza0/cAST-state.git /content/cAST-state
    %cd /content/cAST-state

!pip install -q -r requirements.txt
!pip install -q transformers accelerate editdistance

# Affiche le commit exact actif dans CETTE session — pour ne plus jamais se
# demander "est-ce que le correctif a bien été récupéré ?". Comparez ce hash
# à https://github.com/Robertkiza0/cAST-state/commits/master avant de
# signaler un problème : s'il ne correspond pas au dernier commit affiché
# là-bas, relancez CETTE cellule avant toute autre chose.
print()
!echo "=== Commit actif : $(git rev-parse --short HEAD) — $(git log -1 --format=%s) ==="


## 1bis. (Optionnel) Token Hugging Face

In [ ]:
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN chargé.")
except Exception:
    print("Pas de secret HF_TOKEN configuré — pas grave, pas nécessaire pour un modèle public.")


## 2. Télécharger les données RepoEval (tâches + 8 dépôts réels)

Idempotent : si `data/repos_source/` contient déjà quelque chose, on ne retélécharge rien (sûr à re-exécuter).

In [ ]:
import os
import zipfile

if os.path.isdir('data/repos_source') and os.listdir('data/repos_source'):
    print('RepoEval déjà présent, rien à faire.')
else:
    !rm -rf codet_src
    !git clone --no-checkout --depth 1 https://github.com/microsoft/CodeT.git codet_src
    %cd codet_src
    !git sparse-checkout init --cone
    !git sparse-checkout set RepoCoder
    !git checkout main
    %cd ..

    with zipfile.ZipFile('codet_src/RepoCoder/datasets/datasets.zip') as z:
        z.extractall('datasets rapo')
    with zipfile.ZipFile('codet_src/RepoCoder/repositories/line_and_api_level.zip') as z:
        z.extractall('data/repos_source')

    print('RepoEval : dataset et dépôts extraits.')


## 3. Télécharger les données CrossCodeEval (tâches Python + carte des licences)

Idempotent, même logique. Les vrais dépôts GitHub seront clonés à la volée par `run_benchmark.py` lui-même, dans `cceval_repos/`.

In [ ]:
import os
import tarfile

if os.path.isdir('crosscodeeval_data') and os.listdir('crosscodeeval_data'):
    print('CrossCodeEval déjà présent, rien à faire.')
else:
    !rm -rf cceval_src
    !git clone --no-checkout --depth 1 https://github.com/amazon-science/cceval.git cceval_src
    %cd cceval_src
    !git sparse-checkout init --cone
    !git sparse-checkout set data
    !git checkout main
    %cd ..

    with tarfile.open('cceval_src/data/crosscodeeval_data.tar.xz') as tar:
        tar.extractall('crosscodeeval_data')

    print('CrossCodeEval : tâches extraites.')


## 4. Test minimal du générateur (isolé, SANS le harnais complet)

Charge le modèle et fait UN SEUL appel `generate()` sur un prompt trivial. Objectif : si quelque chose ne va pas avec le chargement du modèle ou la génération elle-même (config du modèle, device_map, VRAM, tokenizer), on le sait en quelques secondes/minutes — pas après avoir attendu une boucle de 80+ tâches sans savoir où ça bloque.

In [ ]:
import time
from generation import HFGenerator

MODEL_NAME = 'bigcode/starcoder2-7b'  # changer ici pour tester CodeLlama, etc.

print(f'Chargement de {MODEL_NAME}...')
t0 = time.time()
gen = HFGenerator(MODEL_NAME, device='cuda')
print(f'Modèle chargé en {time.time()-t0:.1f}s')

print('Test generate()...')
t0 = time.time()
output = gen.generate("def add(a, b):\n    return a + b\n\ndef multiply(a, b):\n    return ")
print(f'generate() en {time.time()-t0:.1f}s')
print(f'Sortie: {output!r}')

print('Deuxième appel (le premier inclut souvent un échauffement CUDA)...')
t0 = time.time()
output2 = gen.generate("def subtract(a, b):\n    return ")
print(f'generate() en {time.time()-t0:.1f}s')
print(f'Sortie: {output2!r}')

del gen
import torch, gc
gc.collect()
torch.cuda.empty_cache()
print('Générateur de test libéré (VRAM/RAM récupérées avant le vrai run).')


## 5. Test rapide, générateur factice (valide tout le pipeline avant le vrai run GPU)

In [ ]:
!python -u run_benchmark.py --dataset repoeval --n-tasks 10 --generator stub


## 6. Run réel — StarCoder2-7B (petit d'abord)

`--n-tasks 20` pour un premier passage rapide (quelques minutes). Une fois que ça tourne bien jusqu'au tableau final, passer à la cellule suivante pour le run complet à l'échelle du projet (`--n-tasks 300`).

`-u` force Python en sortie non bufferisée (en plus du `sys.stdout.reconfigure` déjà dans `run_benchmark.py`) — les lignes de progression détaillées de la tâche 1 doivent apparaître au fur et à mesure, pas d'un coup à la fin.

In [ ]:
!python -u run_benchmark.py --dataset repoeval --n-tasks 20 --generator hf \
    --model-name bigcode/starcoder2-7b --device cuda


## 7. Run complet — StarCoder2-7B, RepoEval + CrossCodeEval

`--tasks-per-repo 38` pour vraiment approcher 300 tâches sur les 8 dépôts RepoEval disponibles localement (300/8 ≈ 38 ; par défaut `--tasks-per-repo 10` plafonne à 80 quel que soit `--n-tasks`).

In [ ]:
!python -u run_benchmark.py --dataset both --n-tasks 300 --tasks-per-repo 38 --generator hf \
    --model-name bigcode/starcoder2-7b --device cuda


## 8. (Optionnel) Run complet — CodeLlama-7B-Python

Même protocole, second générateur : vérifie que le classement des 3 baselines est stable d'un modèle à l'autre (cf. spec — "Conserve le MÊME générateur/LLM ... pour les 3 baselines", ceci compare plutôt across-run comme vérification de robustesse pour le papier).

In [ ]:
!python -u run_benchmark.py --dataset both --n-tasks 300 --tasks-per-repo 38 --generator hf \
    --model-name codellama/CodeLlama-7b-Python-hf --device cuda


## Notes

- Pass@1 == Exact Match ici (pas de harnais d'exécution sur ces variantes line-level de RepoEval/CrossCodeEval — voir `metrics.py:compute_pass_at_1`).
- Le clonage à la volée des dépôts CrossCodeEval peut échouer pour certaines tâches (dépôt supprimé/privé depuis) — `run_benchmark.py` les ignore et continue, en l'indiquant dans la sortie.
- Si une cellule semble bloquée : la tâche 1 de RepoEval affiche maintenant un détail ligne par ligne (chunking/retrieval/generate, par stratégie) — regarder sur QUELLE ligne précise ça s'arrête avant de conclure à un plantage. Un premier `generate()` plus lent que les suivants (échauffement CUDA) est normal, pas un bug.
- En cas de vrai blocage : `Runtime > Restart session` repart toujours d'un état propre ; toutes les cellules d'installation ci-dessus sont idempotentes, donc les réexécuter après un redémarrage ne pose aucun problème (pas de re-téléchargement inutile, pas de clone imbriqué).